# 🏆 エージェント対戦とEloレーティング評価ノートブック

このノートブックは、複数のエージェントを総当たりで対戦させ、Eloレーティングを計算してその結果を出力します。

In [1]:
# ✅ 1. モジュールと関数の準備
import os
import json
from train import AgentFactory, Env_Geister
from geister_game import GeisterGame
import pandas as pd
from collections import defaultdict

# Elo関連
INITIAL_ELO = 1500
K = 32

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating_a, rating_b, score_a, k=K):
    expected_a = expected_score(rating_a, rating_b)
    expected_b = expected_score(rating_b, rating_a)
    new_rating_a = rating_a + k * (score_a - expected_a)
    new_rating_b = rating_b + k * ((1 - score_a) - expected_b)
    return new_rating_a, new_rating_b

In [11]:
# ✅ 2. エージェントの読み込み
agent_info_list = [
    {"id": "Agent-2", "config": "models_geister_agentA/eps2900/config.json", "weights": "models_geister_agentA/eps2900/weights.pth"},
    {"id": "Agent-1", "config": "models_geister_agentA/eps2500/config.json", "weights": "models_geister_agentA/eps2500/weights.pth"},
    {"id": "Agent0", "config": "models_geister_agentA/eps2000/config.json", "weights": "models_geister_agentA/eps2000/weights.pth"},
    {"id": "Agent1", "config": "models_geister_agentA/eps1200/config.json", "weights": "models_geister_agentA/eps1200/weights.pth"},
    {"id": "Agent2", "config": "models_geister_agentA/eps1000/config.json", "weights": "models_geister_agentA/eps1000/weights.pth"},
    {"id": "Agent3", "config": "models_geister_agentA/eps800/config.json", "weights": "models_geister_agentA/eps800/weights.pth"},
    {"id": "Agent4", "config": "models_geister_agentA/eps600/config.json", "weights": "models_geister_agentA/eps600/weights.pth"},
    {"id": "Agent5", "config": "models_geister_agentA/eps400/config.json", "weights": "models_geister_agentA/eps400/weights.pth"},
    {"id": "Agent6", "config": "models_geister_agentA/eps200/config.json", "weights": "models_geister_agentA/eps200/weights.pth"},
    {"id": "Agent7", "config": "models_geister_agentA/eps100/config.json", "weights": "models_geister_agentA/eps100/weights.pth"},
]


agents = {}
for info in agent_info_list:
    with open(info["config"], "r") as f:
        cfg = json.load(f)
    agent = AgentFactory.create_cqc_agent("A", GeisterGame(board_size=4, num_ghosts_per_player=2), cfg, info["weights"])
    agents[info["id"]] = agent

In [12]:
# ✅ 3. 総当たり対戦とElo更新
elo_ratings = defaultdict(lambda: INITIAL_ELO)

for id_a, agent_a in agents.items():
    for id_b, agent_b in agents.items():
        if id_a == id_b:
            continue

        game = GeisterGame(board_size=4, num_ghosts_per_player=2)
        agent_a.player_id, agent_b.player_id = "A", "B"
        agent_a.game, agent_b.game = game, game

        env = Env_Geister(agent_a, agent_b, game)
        winner, _, _ = env.play_one_game_with_log()

        if winner == "A":
            elo_ratings[id_a], elo_ratings[id_b] = update_elo(elo_ratings[id_a], elo_ratings[id_b], 1)
        elif winner == "B":
            elo_ratings[id_a], elo_ratings[id_b] = update_elo(elo_ratings[id_a], elo_ratings[id_b], 0)
        else:
            elo_ratings[id_a], elo_ratings[id_b] = update_elo(elo_ratings[id_a], elo_ratings[id_b], 0.5)

In [14]:
# ✅ 4. 結果の表示
df = pd.DataFrame([
    {"Model": model, "EloRating": round(score, 2)}
    for model, score in elo_ratings.items()
]).sort_values(by="EloRating", ascending=False)
df.reset_index(drop=True, inplace=True)
df

,Model,EloRating
0,Agent4,1558.77
1,Agent2,1549.99
2,Agent1,1539.06
3,Agent7,1532.20
4,Agent-1,1515.73
5,Agent3,1512.99
6,Agent-2,1493.38
7,Agent6,1466.14
8,Agent0,1428.06
9,Agent5,1403.67
